In [14]:
# Install library jika belum ada (uncomment jika perlu)
# !pip install lightgbm pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
import warnings
import json
import joblib

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

In [2]:
# Membaca dataset
df = pd.read_csv('../data/hour.csv')

# Konversi dteday ke tipe datetime
df['dteday'] = pd.to_datetime(df['dteday'])

# Feature Engineering: Menambah fitur tanggal
df['day'] = df['dteday'].dt.day

# Menghapus kolom yang tidak digunakan untuk modeling
# casual dan registered dihapus karena cnt = casual + registered
df_model = df.drop(columns=['instant', 'dteday', 'casual', 'registered'])

print("Data berhasil dimuat. Ukuran dataset:", df_model.shape)
df_model.head()

Data berhasil dimuat. Ukuran dataset: (17379, 14)


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt,day
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16,1
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40,1
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32,1
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13,1
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1,1


In [3]:
# Split data menjadi training dan testing set
# Karena ini data Time Series, kita membagi berdasarkan urutan waktu (bukan random)
# Kita ambil 80% data awal untuk training dan 20% terakhir untuk testing
train_size = int(len(df_model) * 0.8)
train_df = df_model.iloc[:train_size]
test_df = df_model.iloc[train_size:]

X_train = train_df.drop(columns=['cnt'])
y_train = train_df['cnt']
X_test = test_df.drop(columns=['cnt'])
y_test = test_df['cnt']

print(f"Data Training: {X_train.shape}")
print(f"Data Testing: {X_test.shape}")

Data Training: (13903, 13)
Data Testing: (3476, 13)


In [17]:
# Pelatihan Model LightGBM Regressor
# Inisialisasi model LightGBM Regressor
forecasting_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

# Training model dengan Early Stopping
forecasting_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# Save model
joblib.dump(forecasting_model, os.path.join("lgbm_forecast.pkl"))

output_dir = "static"
os.makedirs(output_dir, exist_ok=True)

# Plot Feature Importance
plt.figure(figsize=(10, 6))
lgb.plot_importance(
    forecasting_model,
    importance_type='gain',
    max_num_features=15
)

plt.title('Fitur yang Paling Berpengaruh terhadap Penyewaan')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "feature_importance.png"))
plt.close()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001131 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 316
[LightGBM] [Info] Number of data points in the train set: 13903, number of used features: 13
[LightGBM] [Info] Start training from score 174.639143
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[666]	valid_0's rmse: 62.9255	valid_0's l2: 3959.62


<Figure size 1000x600 with 0 Axes>

In [15]:
# Evaluasi Model
# Prediksi data testing
y_pred = forecasting_model.predict(X_test)

# Menghitung metrik performa
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Hasil Evaluasi Model:")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

metrics = {
    "MAE": round(float(mae), 2),
    "RMSE": round(float(rmse), 2),
    "R2": round(float(r2), 4)
}

# Save metrics
with open("static/forecast_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

# Plot Actual vs Predicted
plt.figure(figsize=(15, 5))
plt.plot(y_test.values[:100], label='Actual', alpha=0.6)
plt.plot(y_pred[:100], label='Predicted', linestyle='--')
plt.title('Actual vs Predicted (First 100 Hours)')
plt.legend()
plt.tight_layout()

# Save image
plt.savefig(os.path.join(output_dir, "forecast_evaluation.png"))
plt.close()

Hasil Evaluasi Model:
MAE  : 40.58
RMSE : 62.93
R2 Score: 0.9185


In [6]:
# Fungsi Prediksi Manual & Forecasting
def predict_manual(tanggal, jam, suhu, kelembaban, kondisi_cuaca, libur):
    """
    Input:
    - tanggal: str 'YYYY-MM-DD'
    - jam: int (0-23)
    - suhu: float (0-1, normalized)
    - kelembaban: float (0-1, normalized)
    - kondisi_cuaca: int (1-4)
    - libur: int (0: No, 1: Yes)
    """
    dt = pd.to_datetime(tanggal)
    year = 0 if dt.year == 2011 else 1 # dataset hanya 2011 (0) & 2012 (1)
    month = dt.month
    weekday = dt.weekday() # 0:Mon, 6:Sun
    day = dt.day
    
    # Penentuan musim sederhana
    if month in [3, 4, 5]: season = 1
    elif month in [6, 7, 8]: season = 2
    elif month in [9, 10, 11]: season = 3
    else: season = 4
    
    workingday = 1 if weekday < 5 and libur == 0 else 0
    atemp = suhu # asumsikan suhu terasa sama dengan suhu riil jika tidak ada input
    windspeed = 0.19 # nilai rata-rata
    
    input_features = pd.DataFrame([[season, year, month, jam, libur, weekday, workingday, kondisi_cuaca, suhu, atemp, kelembaban, windspeed, day]], 
                                  columns=X_train.columns)
    
    prediction = forecasting_model.predict(input_features)[0]
    return max(0, int(prediction))

# Contoh Penggunaan
hasil = predict_manual('2012-06-15', 12, 0.6, 0.4, 1, 0)
print(f"Prediksi jumlah peminjaman: {hasil} sepeda")

Prediksi jumlah peminjaman: 283 sepeda


In [16]:
# Visualisasi Hasil Prediksi Manual dalam Bentuk Line-Bar Chart
# Simulasi Forecasting untuk satu hari penuh
target_date = '2012-08-20'
hours = list(range(24))
forecast_results = []

for h in hours:
    # Menggunakan asumsi cuaca cerah (1) dan suhu sedang (0.5)
    p = predict_manual(target_date, h, 0.5, 0.6, 1, 0)
    forecast_results.append(p)

# Visualisasi Hasil Prediksi Manual (Line-Bar Chart)
plt.figure(figsize=(12, 6))

# Bar Chart
plt.bar(
    hours,
    forecast_results,
    alpha=0.6,
    label='Predicted Count (Bar)'
)

# Line Chart
plt.plot(
    hours,
    forecast_results,
    marker='o',
    linewidth=2,
    label='Predicted Count (Line)'
)

plt.title(f'Bike Rental Forecast for {target_date}')
plt.xlabel('Hour of Day')
plt.ylabel('Predicted Number of Rentals')
plt.xticks(hours)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()

# Save image
file_path = os.path.join(output_dir, "daily_forecast.png")
plt.savefig(file_path)
plt.close()

print(f"Daily forecast plot saved to: {file_path}")

Daily forecast plot saved to: static\daily_forecast.png
